In [21]:
from pathlib import Path

import duckdb
import geopandas as gpd

In [22]:
data_dir = Path("..") / "data"
parquet_glob = str(data_dir / "*.parquet")

con = duckdb.connect()

zones = gpd.read_file(data_dir / "taxi_zones" / "taxi_zones.shp")
zones_lookup = zones[["LocationID", "borough", "zone"]].copy()
zones_lookup["LocationID"] = zones_lookup["LocationID"].astype("int64")
con.register("zones_lookup", zones_lookup)

In [24]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW taxi_clean_2019 AS
SELECT
    t.tpep_pickup_datetime,
    t.tpep_dropoff_datetime,
    t.PULocationID,
    t.DOLocationID,
    pu.borough AS PU_Borough,
    pu.zone AS PU_Zone,
    dz.borough AS DO_Borough,
    dz.zone AS DO_Zone,
    t.trip_distance,
    t.fare_amount,
    t.tip_amount,
    t.total_amount,
    t.pickup_time_hour,
    t.temperature_c,
    t.precipitation_mm,
    t.pickup_business_count,
    t.dropoff_business_count
FROM read_parquet('{parquet_glob}') AS t
LEFT JOIN zones_lookup AS pu
    ON t.PULocationID = pu.LocationID
LEFT JOIN zones_lookup AS dz
    ON t.DOLocationID = dz.LocationID
WHERE t.tpep_pickup_datetime IS NOT NULL
  AND t.tpep_dropoff_datetime IS NOT NULL
  AND t.tpep_dropoff_datetime >= t.tpep_pickup_datetime
  AND t.trip_distance BETWEEN 0 AND 300
  AND t.passenger_count BETWEEN 0 AND 8
  AND t.fare_amount >= 0
  AND t.total_amount BETWEEN 0 AND 1000
  AND t.tpep_pickup_datetime >= DATE '2019-01-01'
  AND t.tpep_pickup_datetime < DATE '2020-01-01'
ORDER BY t.tpep_pickup_datetime
""")

In [29]:
con.sql("DESCRIBE SELECT * FROM taxi_clean_2019").df()

,column_name,column_type,null,key,default,extra
0,tpep_pickup_datetime,TIMESTAMP_NS,YES,None,None,None
1,tpep_dropoff_datetime,TIMESTAMP_NS,YES,None,None,None
2,PULocationID,BIGINT,YES,None,None,None
3,DOLocationID,BIGINT,YES,None,None,None
4,PU_Borough,VARCHAR,YES,None,None,None
5,PU_Zone,VARCHAR,YES,None,None,None
6,DO_Borough,VARCHAR,YES,None,None,None
7,DO_Zone,VARCHAR,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [25]:
con.execute("""
CREATE OR REPLACE TEMP VIEW stream_trips AS
SELECT *
FROM taxi_clean_2019
WHERE PU_Borough IN ('Manhattan', 'Brooklyn', 'Queens')
  AND DO_Borough IN ('Manhattan', 'Brooklyn', 'Queens')
""")

In [26]:
con.sql("SELECT * FROM stream_trips LIMIT 5").df()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,PU_Borough,PU_Zone,DO_Borough,DO_Zone,trip_distance,fare_amount,tip_amount,total_amount,pickup_time_hour,temperature_c,precipitation_mm,pickup_business_count,dropoff_business_count
0,2019-01-01 00:00:01,2019-01-01 00:05:17,263,74,Manhattan,Yorkville West,Manhattan,East Harlem North,1.73,7.0,0.00,8.30,2019-01-01,6.4,1.8,165,288
1,2019-01-01 00:00:03,2019-01-01 00:04:26,80,112,Brooklyn,East Williamsburg,Brooklyn,Greenpoint,0.60,5.0,0.00,6.30,2019-01-01,6.4,1.8,251,285
2,2019-01-01 00:00:05,2019-01-01 00:11:05,231,148,Manhattan,TriBeCa/Civic Center,Manhattan,Lower East Side,1.53,9.0,0.00,10.30,2019-01-01,6.4,1.8,205,223
3,2019-01-01 00:00:06,2019-01-01 00:44:05,114,79,Manhattan,Greenwich Village South,Manhattan,East Village,3.20,26.0,5.45,32.75,2019-01-01,6.4,1.8,77,256
4,2019-01-01 00:00:09,2019-01-01 00:18:03,161,209,Manhattan,Midtown Center,Manhattan,Seaport,6.45,21.0,4.46,26.76,2019-01-01,6.4,1.8,562,35


In [27]:
con.execute("""
CREATE OR REPLACE TEMP VIEW september_trips AS
SELECT *
FROM stream_trips
WHERE EXTRACT(MONTH FROM tpep_pickup_datetime) = 9
""")

In [28]:
row_count = con.sql("SELECT COUNT(*) FROM september_trips").fetchone()[0]
row_count

6370989

In [30]:
con.sql("""
SELECT COUNT(*)
FROM september_trips
WHERE tpep_pickup_datetime < DATE '2019-09-03'
""").fetchone()[0]

273852

In [31]:
con.execute("""
COPY (
    SELECT *
    FROM september_trips
    WHERE tpep_pickup_datetime < DATE '2019-09-03'
) TO '../data/september_first_two_days.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")